In [2]:
import pandas as pd
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support
import joblib
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

In [3]:
fenotipo = pd.read_csv('banco_fenotipos_conformal.csv')
imcs = [c for c in fenotipo.columns if 'imc' in c.lower() or 'obesidade' in c.lower() or 'obes_central' in c.lower() or 'hdl' in c.lower()]
fenotipo = fenotipo.drop(columns=imcs)

In [4]:
fenotipo = fenotipo.dropna()

In [5]:
# Filtra todas as colunas que possuem esses termos no nome
cols_para_remover = [c for c in fenotipo.columns if 'glicose' in c.lower() or 'homa' in c.lower() or 'diabetes' in c.lower()]
cols_para_remover += ['faixa_etaria','imc', 'imc_cat4_aferido', 'imc_cat_aferido', 'ep', 'id', 'samplefilename', 'ccm', 'obes_central', 'cc_estat', 'cc_estat_cat', 'medDM_bioq','medHAS_bioq', 'has2','colnaohdl_cat', 'CP1', 'CP2', 'CP3', 'CP4','CP5', 'CP6']
X = fenotipo.drop(columns=cols_para_remover, errors='ignore')
y = fenotipo['diabetes2']

In [6]:
colunas_categoricas = ['fumo', 'raca_cor', 'conjugal3', 'trabalho2', 'bebem2']
X = pd.get_dummies(X, columns=colunas_categoricas, drop_first=True, dtype=int)

In [7]:
modelorf = RandomForestClassifier(n_estimators=1000, random_state=2, n_jobs=-1, max_features=0.3, oob_score=True, class_weight='balanced', verbose = 1)

In [8]:
modelorf.fit(X,y)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.4s
[Parallel(n_jobs=-1)]: Done 776 tasks      | elapsed:    0.8s
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:    1.0s finished


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",1000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.3
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [9]:
modelorf.score(X,y)

thr = 0.471

y_pred = modelorf.oob_decision_function_[:, 1] > thr

precision_recall_fscore_support(y, y_pred)

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 426 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 776 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 1000 out of 1000 | elapsed:    0.1s finished


(array([0.85500747, 0.33333333]),
 array([0.97278912, 0.07619048]),
 array([0.91010342, 0.12403101]),
 array([588, 105]))

In [10]:
# Pega as importâncias matemáticas do modelo
importancias = modelorf.feature_importances_

# Pega o nome das 244 mil colunas do seu X
nomes_variaveis = X.columns

# Cria uma tabela bonita ligando o nome ao valor de importância
tabela_importancia = pd.DataFrame({
    'Variavel': nomes_variaveis,
    'Importancia': importancias
})
top_20 = tabela_importancia.sort_values(by='Importancia', ascending=False).head(20)
print(top_20)

                          Variavel  Importancia
2                            idade     0.280822
3                        pas_final     0.144262
0                          hb_g_dl     0.136709
4                        pad_final     0.125880
16              conjugal3_solteiro     0.059912
18             trabalho2_estudante     0.030706
6                       fumo_nunca     0.025179
19         trabalho2_nÃ£o trabalha     0.024407
1                             sexo     0.020740
5                     fumo_fumante     0.018692
8                  raca_cor_branca     0.015875
20              trabalho2_trabalha     0.015851
22           bebem2_atÃ© 3x semana     0.014622
14    conjugal3_casado/un. estavel     0.014230
23                bebem2_nÃ£o bebe     0.013897
17                 conjugal3_viuvo     0.013688
12                  raca_cor_parda     0.012873
15  conjugal3_separado/desq/divorc     0.009393
13                  raca_cor_preta     0.008186
11                 raca_cor_outras     0

In [11]:
# Salvar modelos em arquivo pkl
joblib.dump(modelorf, 'modelo1.pkl')

['modelo1.pkl']

In [12]:
import numpy as np

# 1. Re-run decision_path but unpack BOTH variables this time
# (Notice we pass the row slice X.iloc[0:1] to keep it 2D)
node_indicator, estimators_indices = modelorf.decision_path(X.iloc[0:1])

# 2. Set up the constants for our single sample
sample_id = 0  
feature_names = X.columns
sample_values = X.iloc[sample_id]

# 3. Extract the raw continuous node indices visited by this sample
node_index = node_indicator.indices[
    node_indicator.indptr[sample_id] : node_indicator.indptr[sample_id + 1]
]

# 4. Loop through every tree using the safe 'estimators_indices' array
for tree_id, estimator in enumerate(modelorf.estimators_):
    # Safe and correct way to get the matrix boundaries for this specific tree
    node_offset = estimators_indices[tree_id]
    next_node_offset = estimators_indices[tree_id + 1]
    
    print(f"\n=====================================")
    print(f"--- TREE {tree_id} DECISION PATH ---")
    print(f"=====================================")
    
    # Filter the nodes that belong to this specific tree
    for node_id in node_index:
        if node_offset <= node_id < next_node_offset:
            # Normalize the node ID to match this specific tree's local index
            local_node_id = node_id - node_offset
            
            # Access the underlying tree structure details
            tree_ = estimator.tree_
            feature_idx = tree_.feature[local_node_id]
            
            # If the feature index is -1, it means this node is a final Leaf Node
            if feature_idx == -1:
                predicted_value = tree_.value[local_node_id]
                print(f"[Leaf Node {local_node_id}] Prediction distribution/class: {predicted_value}")
            else:
                # Get the threshold and feature name for the split
                threshold = tree_.threshold[local_node_id]
                feat_name = feature_names[feature_idx]
                actual_val = sample_values[feat_name]
                
                # Determine which direction the data went
                direction = "<=" if actual_val <= threshold else ">"
                
                print(f"[Node {local_node_id}] Split on: {feat_name}")
                print(f"         Rule: {feat_name} <= {threshold:.4f}")
                print(f"         Your Data: {actual_val} ({direction} {threshold:.4f})")
                print(f"-------------------------------------")


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 776 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:    0.1s finished



--- TREE 0 DECISION PATH ---
[Node 0] Split on: trabalho2_estudante
         Rule: trabalho2_estudante <= 0.5000
         Your Data: 0.0 (<= 0.5000)
-------------------------------------
[Node 1] Split on: trabalho2_nÃ£o trabalha
         Rule: trabalho2_nÃ£o trabalha <= 0.5000
         Your Data: 0.0 (<= 0.5000)
-------------------------------------
[Node 2] Split on: conjugal3_viuvo
         Rule: conjugal3_viuvo <= 0.5000
         Your Data: 0.0 (<= 0.5000)
-------------------------------------
[Node 3] Split on: idade
         Rule: idade <= 49.5000
         Your Data: 58.0 (> 49.5000)
-------------------------------------
[Node 21] Split on: hb_g_dl
         Rule: hb_g_dl <= 16.8000
         Your Data: 20.5 (> 16.8000)
-------------------------------------
[Node 79] Split on: bebem2_atÃ© 3x semana
         Rule: bebem2_atÃ© 3x semana <= -2.0000
         Your Data: 0.0 (> -2.0000)
-------------------------------------

--- TREE 1 DECISION PATH ---
[Node 0] Split on: idade
        

In [26]:
import numpy as np

arvore = modelorf.estimators_[0]
indice_da_amostra = 1

X_numpy = X.values 
amostra_alvo = X_numpy[[indice_da_amostra]]

caminhos_todos = arvore.decision_path(X_numpy).toarray()
caminho_alvo = caminhos_todos[indice_da_amostra]

print(f"--- Comparando Amostra {indice_da_amostra} com as demais ---\n")

for i in range(len(X_numpy)):
    if i == indice_da_amostra:
        continue 
    
    caminho_comparado = caminhos_todos[i]
    
    nos_em_comum = np.sum((caminho_alvo == 1) & (caminho_comparado == 1))
    
    total_nos_alvo = np.sum(caminho_alvo)
    
    similaridade = (nos_em_comum / total_nos_alvo) * 100
    
    if similaridade >= 90.0:
        amostra_atual = X_numpy[[i]]
        mesmo_no_final = arvore.apply(amostra_alvo)[0] == arvore.apply(amostra_atual)[0]
        
        print(f"A amostra {i} tem {similaridade:.1f}% do caminho igual à amostra alvo!")
        print(f"   Ambas terminaram no mesmo nó folha? {mesmo_no_final}") if mesmo_no_final else None
        print()


--- Comparando Amostra 1 com as demais ---

A amostra 30 tem 90.9% do caminho igual à amostra alvo!

A amostra 188 tem 90.9% do caminho igual à amostra alvo!

A amostra 199 tem 90.9% do caminho igual à amostra alvo!

A amostra 392 tem 90.9% do caminho igual à amostra alvo!

A amostra 595 tem 90.9% do caminho igual à amostra alvo!

